In [18]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [19]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [20]:
Path("../outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("../outputs/models").mkdir(parents=True, exist_ok=True)

In [21]:
with open("../data/graph/train_graph.pkl", "rb") as f:
    train_graph = pickle.load(f)

with open("../data/graph/test_graph.pkl", "rb") as f:
    test_graph = pickle.load(f)

with open("../data/graph/wallet_mapping.pkl", "rb") as f:
    wallet_mapping = pickle.load(f)

num_nodes = wallet_mapping["num_nodes"]

print("Num nodes:", num_nodes)
print("Train edges:", train_graph["edge_index"].shape)
print("Test edges:", test_graph["edge_index"].shape)

Num nodes: 108582
Train edges: (2, 38454)
Test edges: (2, 298160)


In [22]:
class EdgeDataset(Dataset):
    def __init__(self, graph):
        self.edge_index = torch.tensor(graph["edge_index"], dtype=torch.long)
        self.edge_features = torch.tensor(graph["edge_features"], dtype=torch.float32)
        self.labels = torch.tensor(graph["edge_labels"], dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        source = self.edge_index[0, idx]
        target = self.edge_index[1, idx]
        edge_feat = self.edge_features[idx]
        label = self.labels[idx]

        return source, target, edge_feat, label

In [23]:
BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_graph)
test_dataset = EdgeDataset(test_graph)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [24]:
class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)

        input_dim = embedding_dim * 2 + edge_feat_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        tgt_emb = self.node_embedding(target)

        x = torch.cat([src_emb, tgt_emb, edge_feat], dim=1)

        logits = self.mlp(x).squeeze(1)

        return logits

In [25]:
edge_feat_dim = train_graph["edge_features"].shape[1]

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(108582, 64)
  (mlp): Sequential(
    (0): Linear(in_features=138, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [26]:
labels = train_graph["edge_labels"]

num_positive = np.sum(labels == 1)
num_negative = np.sum(labels == 0)

pos_weight_value = num_negative / num_positive

print("Positive:", num_positive)
print("Negative:", num_negative)
print("pos_weight:", pos_weight_value)

Positive: 754
Negative: 37700
pos_weight: 50.0


In [27]:
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [28]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [29]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [30]:
def evaluate(model, loader, threshold=0.5):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(
                source,
                target,
                edge_feat
            )

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                label.numpy()
            )

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, preds)
    }

    return metrics

In [31]:
EPOCHS = 20

history = []

best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_metrics = evaluate(model, test_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | "
        f"Rec: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f} | "
        f"ROC-AUC: {test_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {test_metrics['pr_auc']:.4f}"
    )

    if test_metrics["f1"] > best_f1:
        best_f1 = test_metrics["f1"]

        torch.save(
            model.state_dict(),
            "../outputs/models/graph_class_weight_best.pt"
        )

Epoch 01 | Loss: 1.2518 | Acc: 0.8348 | Prec: 0.0005 | Rec: 0.3594 | F1: 0.0009 | ROC-AUC: 0.6371 | PR-AUC: 0.0025
Epoch 02 | Loss: 1.0649 | Acc: 0.7179 | Prec: 0.0004 | Rec: 0.5625 | F1: 0.0009 | ROC-AUC: 0.7104 | PR-AUC: 0.0016
Epoch 03 | Loss: 0.9575 | Acc: 0.7626 | Prec: 0.0005 | Rec: 0.5781 | F1: 0.0010 | ROC-AUC: 0.7454 | PR-AUC: 0.0016
Epoch 04 | Loss: 0.8256 | Acc: 0.7899 | Prec: 0.0006 | Rec: 0.5469 | F1: 0.0011 | ROC-AUC: 0.7491 | PR-AUC: 0.0021
Epoch 05 | Loss: 0.7061 | Acc: 0.8199 | Prec: 0.0006 | Rec: 0.5156 | F1: 0.0012 | ROC-AUC: 0.7455 | PR-AUC: 0.0049
Epoch 06 | Loss: 0.5744 | Acc: 0.8134 | Prec: 0.0006 | Rec: 0.5000 | F1: 0.0011 | ROC-AUC: 0.7492 | PR-AUC: 0.0186
Epoch 07 | Loss: 0.4584 | Acc: 0.8571 | Prec: 0.0005 | Rec: 0.3594 | F1: 0.0011 | ROC-AUC: 0.7469 | PR-AUC: 0.0712
Epoch 08 | Loss: 0.3637 | Acc: 0.8454 | Prec: 0.0006 | Rec: 0.4375 | F1: 0.0012 | ROC-AUC: 0.7590 | PR-AUC: 0.0336
Epoch 09 | Loss: 0.2956 | Acc: 0.8883 | Prec: 0.0006 | Rec: 0.3281 | F1: 0.0013 

In [32]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../outputs/metrics/graph_class_weight_history.csv",
    index=False
)

history_df

,epoch,train_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,1.251776,0.834763,0.000467,0.359375,0.000933,0.637082,0.002471
1,2,1.064939,0.717927,0.000428,0.562500,0.000855,0.710421,0.001588
2,3,0.957483,0.762641,0.000523,0.578125,0.001045,0.745383,0.001641
3,4,0.825595,0.789878,0.000559,0.546875,0.001116,0.749149,0.002110
4,5,0.706099,0.819909,0.000615,0.515625,0.001228,0.745478,0.004871
5,6,0.574378,0.813362,0.000575,0.500000,0.001149,0.749247,0.018625
6,7,0.458443,0.857063,0.000540,0.359375,0.001078,0.746871,0.071215
7,8,0.363692,0.845432,0.000608,0.437500,0.001214,0.758957,0.033601
8,9,0.295644,0.888298,0.000631,0.328125,0.001259,0.756399,0.070826
9,10,0.214232,0.916146,0.000761,0.296875,0.001518,0.774562,0.070256


In [33]:
model.load_state_dict(
    torch.load("../outputs/models/graph_class_weight_best.pt")
)

final_metrics = evaluate(model, test_loader)

print("Final Metrics:")
for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(k, ":", v)

print("Confusion Matrix:")
print(final_metrics["confusion_matrix"])

Final Metrics:
threshold : 0.5
accuracy : 0.9652938019855112
precision : 0.0013576415826221878
recall : 0.21875
f1 : 0.0026985350809560524
roc_auc : 0.779439420312584
pr_auc : 0.06893288226457704
Confusion Matrix:
[[287798  10298]
 [    50     14]]


In [34]:
final_result = {
    "model": "graph_class_weight",
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": final_metrics["confusion_matrix"][0, 0],
    "fp": final_metrics["confusion_matrix"][0, 1],
    "fn": final_metrics["confusion_matrix"][1, 0],
    "tp": final_metrics["confusion_matrix"][1, 1],
}

pd.DataFrame([final_result]).to_csv(
    "../outputs/metrics/graph_class_weight_final.csv",
    index=False
)

final_result

{'model': 'graph_class_weight',
 'accuracy': 0.9652938019855112,
 'precision': 0.0013576415826221878,
 'recall': 0.21875,
 'f1': 0.0026985350809560524,
 'roc_auc': 0.779439420312584,
 'pr_auc': 0.06893288226457704,
 'tn': np.int64(287798),
 'fp': np.int64(10298),
 'fn': np.int64(50),
 'tp': np.int64(14)}